# ✈️ Air Passenger Time-Series Forecasting

This notebook analyses the monthly AirPassengers dataset from 1949 to 1960. It covers data validation, trend and seasonal diagnostics, stationarity, chronological validation, baseline forecasts, seasonal exponential smoothing, SARIMA, residual analysis, and a future forecast.

> **Scope:** This is a historical teaching dataset. The model outputs are educational examples, not operational airline demand guidance.

## 1. Environment

Install the project's dependencies from the repository root using `python -m pip install -r requirements.txt`.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("default")
DATA_PATH = Path("AirPassengers.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("Run this notebook from the repository root so AirPassengers.csv is found.")

raw = pd.read_csv(DATA_PATH)
required = {"Month", "Passengers"}
missing = required.difference(raw.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

raw["Month"] = pd.to_datetime(raw["Month"], dayfirst=True, errors="coerce")
raw["Passengers"] = pd.to_numeric(raw["Passengers"], errors="coerce")
if raw[["Month", "Passengers"]].isna().any().any():
    raise ValueError("Dataset contains invalid dates or passenger counts.")

series = raw.set_index("Month")["Passengers"].sort_index()
series.index = series.index.to_period("M").to_timestamp(how="start")
series = series.asfreq("MS")
if series.isna().any() or series.index.has_duplicates:
    raise ValueError("Expected a continuous, unique monthly time series.")
series.name = "Passengers"
series.head()

## 2. Inspect the data

Confirm the number of observations, monthly coverage, missing values, and descriptive statistics before training.

In [ ]:
display(series.to_frame().describe())
print(f"Observations: {len(series)}")
print(f"Date range: {series.index.min():%Y-%m} to {series.index.max():%Y-%m}")
print(f"Missing values: {series.isna().sum()}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
series.plot(ax=ax)
ax.set(title="Monthly international airline passengers", xlabel="Month", ylabel="Passengers (thousands)")
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Trend and seasonality

The data has a rising level and repeating within-year structure. Classical multiplicative decomposition is shown as a diagnostic because seasonal variation grows as the series level increases.

In [ ]:
decomposition = seasonal_decompose(series, model="multiplicative", period=12, extrapolate_trend="freq")
fig = decomposition.plot()
fig.set_size_inches(12, 8)
plt.tight_layout()
plt.show()

## 4. Stationarity diagnostics

The Augmented Dickey-Fuller test has a unit-root null hypothesis. A low p-value is evidence against that null, but the test should be interpreted alongside plots and model residuals. Seasonal differencing is a common transformation for a series with annual seasonality.

In [ ]:
adf_original = adfuller(series, autolag="AIC")
print(f"ADF statistic: {adf_original[0]:.4f}")
print(f"ADF p-value: {adf_original[1]:.6f}")

log_series = np.log(series)
seasonal_difference = log_series.diff(12).dropna()
adf_seasonal = adfuller(seasonal_difference, autolag="AIC")
print(f"12-month differenced ADF statistic: {adf_seasonal[0]:.4f}")
print(f"12-month differenced ADF p-value: {adf_seasonal[1]:.6f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
log_series.plot(ax=axes[0], title="Log passenger series")
seasonal_difference.plot(ax=axes[1], title="Log series after 12-month differencing")
plt.tight_layout()
plt.show()

## 5. Chronological train/test split

Do not shuffle time-series observations. The final 24 months are kept as a holdout period. All model selection and fitting below use only the earlier training observations.

In [ ]:
test_size = 24
train = series.iloc[:-test_size]
test = series.iloc[-test_size:]

print(f"Training: {train.index.min():%Y-%m} to {train.index.max():%Y-%m} ({len(train)} observations)")
print(f"Testing:  {test.index.min():%Y-%m} to {test.index.max():%Y-%m} ({len(test)} observations)")

## 6. Establish simple baselines

Compare complex models against a naive forecast and a seasonal-naive forecast that repeats the last available annual cycle.

In [ ]:
naive_forecast = pd.Series(train.iloc[-1], index=test.index, name="Naive")
last_year = train.iloc[-12:].to_numpy()
seasonal_naive_forecast = pd.Series(
    [last_year[i % 12] for i in range(len(test))],
    index=test.index,
    name="Seasonal naive",
)

## 7. Fit seasonal forecasting models

The seasonal period is 12 for monthly observations. Both models are fitted only on the training portion.

In [ ]:
ets_fit = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

ets_forecast = pd.Series(
    ets_fit.forecast(len(test)).to_numpy(),
    index=test.index,
    name="Seasonal Exponential Smoothing",
)

In [ ]:
sarima_fit = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

sarima_forecast = pd.Series(
    sarima_fit.forecast(len(test)).to_numpy(),
    index=test.index,
    name="SARIMA",
)

## 8. Evaluate the holdout period

MAE and RMSE use the passenger-count units. MAPE is supplementary and should not be used alone. R-squared is not the primary forecast metric here.

In [ ]:
def evaluate(actual: pd.Series, predicted: pd.Series) -> dict[str, float]:
    aligned = pd.concat(
        [actual.rename("actual"), predicted.rename("predicted")], axis=1
    ).dropna()
    if aligned.empty:
        raise ValueError("No overlapping actual and forecast values.")
    return {
        "MAE": float(mean_absolute_error(aligned["actual"], aligned["predicted"])),
        "RMSE": float(np.sqrt(mean_squared_error(aligned["actual"], aligned["predicted"]))),
        "MAPE (%)": float(
            (np.abs((aligned["actual"] - aligned["predicted"]) / aligned["actual"])).mean() * 100
        ),
    }

forecasts = {
    "Naive": naive_forecast,
    "Seasonal naive": seasonal_naive_forecast,
    "Seasonal Exponential Smoothing": ets_forecast,
    "SARIMA": sarima_forecast,
}
results = pd.DataFrame(
    {name: evaluate(test, prediction) for name, prediction in forecasts.items()}
).T.sort_values("RMSE")

display(results)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
train.plot(ax=ax, label="Training data")
test.plot(ax=ax, label="Held-out actuals", linewidth=2)
for name, prediction in forecasts.items():
    prediction.plot(ax=ax, label=name, alpha=0.85)
ax.set(title="Chronological holdout forecast comparison", xlabel="Month", ylabel="Passengers (thousands)")
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Residual diagnostics

Residual plots can expose remaining patterns or model misspecification. A good-looking residual plot is not proof of generalization.

In [ ]:
residuals = sarima_fit.resid.dropna()
fig, axes = plt.subplots(2, 1, figsize=(12, 7))
residuals.plot(ax=axes[0], title="SARIMA residuals")
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].set_title("Residual distribution")
plt.tight_layout()
plt.show()

## 10. Refit and forecast future months

After evaluating candidates on the chronological holdout, refit the selected approach on the full historical series. This illustrative cell uses seasonal exponential smoothing for a 12-month horizon.

In [ ]:
forecast_horizon = 12
final_ets = ExponentialSmoothing(
    series,
    trend="add",
    seasonal="add",
    seasonal_periods=12,
    initialization_method="estimated",
).fit(optimized=True)

future_index = pd.date_range(
    series.index[-1] + pd.offsets.MonthBegin(1),
    periods=forecast_horizon,
    freq="MS",
)
future_forecast = pd.Series(
    final_ets.forecast(forecast_horizon).to_numpy(),
    index=future_index,
    name="Forecast",
)
display(future_forecast.to_frame())

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
series.plot(ax=ax, label="Historical")
future_forecast.plot(ax=ax, label="Illustrative forecast", color="tab:red")
ax.set(title="Illustrative 12-month passenger forecast", xlabel="Month", ylabel="Passengers (thousands)")
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 11. Conclusions and limitations

- The historical monthly series contains a clear upward trend and annual seasonality.
- Forecast evaluation must preserve chronological ordering.
- Seasonal-naive baselines are important for this dataset.
- Model rankings depend on the holdout period and model specification.
- The CSV contains only passenger counts, not fares, routes, capacity, economic conditions, disruptions, or airline-specific features.
- A production system would require current data, rolling-origin validation, prediction intervals, monitoring, retraining procedures, and domain review.

No claim is made that one candidate is universally best.